# TSFS12 Hand-in exercise 4: Collaborative Control 5.1

In [1]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.animation as animation
from collab_functions import CreateAgent, AgentFormation

It is necessary to plot in an external windows for the animations to work.
Run the ipython-command below to activate plotting in external windows.

If you have no default Matplotlib backend, you might have to write

```%matplotlib tk``` 

to explicitly state which backend to use (tk is a good choice that should work on all platforms)

In [2]:
%matplotlib qt

ImportError: Failed to import any of the following Qt binding modules: PyQt6, PySide6, PyQt5, PySide2

# Model, controllers, and agent formation
A first example are 4 agents, all modeled by a single integrator, and all measuring their absolute position.  The controller for each agent is a proportional controller.

In [ ]:
def double_integrator(t, x, u, model_params):
    '''
    System-model functions
     
     Input arguments are:
     t - time 
     x - agent state
     u - the control input u, 
     model_params - structure with parameters of the model '''

    velocity = x[2:4]
    return np.concatenate((velocity, u))

def controller_p_v(y, y_ref, ctrl_params):
    ''' 
    Control function   

    Output: Control signal 
    '''
    Kp = ctrl_params["kp"]
    Kv = ctrl_params["kv"]

    p, v = y[:2], y[2:]
    p_ref, v_ref = y_ref[:2], y_ref[2:]

    return Kp * (p_ref - p) + Kv * (v_ref - v)


def controller_relative(y, y_ref, ctrl_params):
    Kp = ctrl_params["kp"]
    Kv = ctrl_params["kv"]

    u = np.zeros(2)

    for k in range(len(y)):
        dp = y[k, :2]
        dv = y[k, 2:4]

        dp_ref = y_ref[k, :2]
        dv_ref = y_ref[k, 2:4]

        u += Kp * (dp - dp_ref) + Kv * (dv - dv_ref)

    return u

Now, formulate the formation graph and position references. The graph is specified by a set of edges. An edge (i, j) represents that agent j meauseras the distance from agent i. If agent i measures its own absolute position, this is represented by a tuple with element i as (i,) and this is the case for all agents in this example.

In [ ]:
n = 4  # (px, py, vx, vy)
G = [
    (0,),       # Agent 0 measures its own absolute state

    (0, 1),     # Agent 1 measures agent 0
    (2, 1),     # Agent 1 measures agent 2

    (1, 2),     # Agent 2 measures agent 1
    (3, 2),     # Agent 2 measures agent 3

    (2, 3),
    (4, 3),

    (3, 4),
    (5, 4),

    (4, 5),
    (6, 5),

    (5, 6),
    (7, 6),

    (6, 7),     # Bottom agent only has one adjacent neighbour
]

formation_references = [
    # Agent 0: absolute reference
    lambda t: np.array([0, 4, 0, 0]),

    # Agent 1: relative to agents 0 and 2
    lambda t: np.array([
        [0,  1, 0, 0],
        [0, -1, 0, 0]
    ]),

    # Agent 2: relative to agents 1 and 3
    lambda t: np.array([
        [0,  1, 0, 0],
        [0, -1, 0, 0]
    ]),

    # Agent 3
    lambda t: np.array([
        [0,  1, 0, 0],
        [0, -1, 0, 0]
    ]),

    # Agent 4
    lambda t: np.array([
        [0,  1, 0, 0],
        [0, -1, 0, 0]
    ]),

    # Agent 5
    lambda t: np.array([
        [0,  1, 0, 0],
        [0, -1, 0, 0]
    ]),

    # Agent 6
    lambda t: np.array([
        [0,  1, 0, 0],
        [0, -1, 0, 0]
    ]),

    # Agent 7 only sees agent 6
    lambda t: np.array([
        [0, 1, 0, 0]
    ]),
]

Create all agents by assigning model and controller using ```CreateAgent``` and then form the formation using ```AgentFormation``` by providing the references.

# Simulate and animate
Define an initial state and evaluate the formatation state-transition function

In [ ]:
x0 = np.array([
    1, 0, 0, 0,
    2, 0, 0, 0,
    3, 0, 0, 0,
    4, 0, 0, 0,
    5, 0, 0, 0,
    6, 0, 0, 0,
    7, 0, 0, 0,
    8, 0, 0, 0,
])

Simulate the formation

In [158]:
t = np.arange(0, 10, 0.05) # Time vector for the simulation
x = odeint(formation.ode, x0, t)

Animate the results

In [ ]:
fig, ax = plt.subplots(num=20, clear=True, figsize=(8, 6))
ax.axis([-2, 9, -4, 5])
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Exercise 5.2.1: Displacement-based formation control")

# One moving marker and one trajectory line per agent
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:orange", "tab:purple", "tab:brown", "tab:pink"]
agents_plot = [ax.plot([], [], "o", color=colors[i])[0] for i in range(len(agents))]
trails = [ax.plot([], [], "-", color=colors[i], alpha=0.7)[0] for i in range(len(agents))]
time_text = ax.text(0.02, 0.95, "", transform=ax.transAxes)
params_text = ax.text(
    0.02, 0.02,
    f"Kp = {ctrl_params['kp']}, Kv = {ctrl_params['kv']}\n"
    transform=ax.transAxes,
    ha="left", va="bottom",
)

def animate(i):
    for idx, (marker, trail) in enumerate(zip(agents_plot, trails)):
        px = x[: i + 1, n * idx]
        py = x[: i + 1, n * idx + 1]
        marker.set_data([px[-1]], [py[-1]])
        trail.set_data(px, py)
    time_text.set_text(f"t = {t[i]:.1f}")
    return tuple(agents_plot + trails) + (time_text, params_text)

ani = animation.FuncAnimation(
    fig, animate, interval=20, frames=x.shape[0], blit=True, repeat=False
)
